In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [2]:
# Input path and load .csv files
input_path = '../data/processed/lea_features_v2.csv'
df = pd.read_csv(input_path)
df.head()

,LEAID,absent_rate,log_enrolled,grad_rate
0,100005,13.427627,8.602269,94.0
1,100006,16.286416,8.647871,91.0
2,100007,9.256037,9.572828,94.0
3,100008,9.866716,9.287672,96.0
4,100011,11.889492,7.614312,95.0


In [3]:
# Confirm data was loaded correctly
print(df.shape)
print(df.describe().round(2))
print('NaNs:', df.isna().sum().sum())

(5940, 4)
            LEAID  absent_rate  log_enrolled  grad_rate
count     5940.00      5940.00       5940.00    5940.00
mean   3003036.81        15.55          8.19      88.12
std    1488087.47         9.85          0.97       9.98
min     100005.00         0.00          3.14       1.00
25%    1807890.00         9.30          7.53      86.00
50%    3302485.00        13.67          8.03      92.00
75%    4203585.00        19.22          8.69      95.00
max    7200030.00        96.58         13.14      99.00
NaNs: 0


## Train/Test Split

In [4]:
X = df[['absent_rate', 'log_enrolled']]
y = df['grad_rate']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Baseline: Linear Regression

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

y_pred = model.predict(X_test)

Intercept: 95.63061201740912
Coefficients: [-0.62399045  0.26772666]


In [6]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 2))

MAE: 5.05
RMSE: 8.22
R²: 0.29


## Random Forest (Tuned via GridSearchCV)

In [7]:
param_grid = {'max_depth': [3, 5, 8, None]}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5
)

grid_search.fit(X_train, y_train)

print("Best max_depth:", grid_search.best_params_)
best_rf = grid_search.best_estimator_

Best max_depth: {'max_depth': 8}


In [8]:
y_pred_rf_tuned = best_rf.predict(X_test)

mae_rf_tuned = mean_absolute_error(y_test, y_pred_rf_tuned)
rmse_rf_tuned = root_mean_squared_error(y_test, y_pred_rf_tuned)
r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned)

print("Tuned RF MAE:", round(mae_rf_tuned, 2))
print("Tuned RF RMSE:", round(rmse_rf_tuned, 2))
print("Tuned RF R²:", round(r2_rf_tuned, 2))

Tuned RF MAE: 5.08
Tuned RF RMSE: 8.19
Tuned RF R²: 0.29


## Segment Analysis

8% threshold based on the flat-to-declining breakpoint identified in notebook 04 EDA (Figure 2).

In [9]:
# Boolean mask for the split point in the test set
low_mask = X_test['absent_rate'] < 8
high_mask = X_test['absent_rate'] >= 8

# Segment actuals
y_test_low = y_test[low_mask]
y_test_high = y_test[high_mask]

# Segment predictions — linear
y_pred_low = y_pred[low_mask]
y_pred_high = y_pred[high_mask]

# Segment predictions — random forest
y_pred_rf_low = y_pred_rf_tuned[low_mask]
y_pred_rf_high = y_pred_rf_tuned[high_mask]

# Metrics per segment
print("Low absenteeism (<8%) — n =", low_mask.sum())
print("  Linear MAE:", round(mean_absolute_error(y_test_low, y_pred_low), 2))
print("  Linear RMSE:", round(root_mean_squared_error(y_test_low, y_pred_low), 2))
print("  RF MAE:", round(mean_absolute_error(y_test_low, y_pred_rf_low), 2))
print("  RF RMSE:", round(root_mean_squared_error(y_test_low, y_pred_rf_low), 2))

print("High absenteeism (>=8%) — n =", high_mask.sum())
print("  Linear MAE:", round(mean_absolute_error(y_test_high, y_pred_high), 2))
print("  Linear RMSE:", round(root_mean_squared_error(y_test_high, y_pred_high), 2))
print("  RF MAE:", round(mean_absolute_error(y_test_high, y_pred_rf_high), 2))
print("  RF RMSE:", round(root_mean_squared_error(y_test_high, y_pred_rf_high), 2))

# Check that no rows were dropped between segments
print("Total accounted for:", low_mask.sum() + high_mask.sum(), "vs. len(X_test):", len(X_test))

Low absenteeism (<8%) — n = 214
  Linear MAE: 4.13
  Linear RMSE: 9.94
  RF MAE: 4.17
  RF RMSE: 9.1
High absenteeism (>=8%) — n = 974
  Linear MAE: 5.26
  Linear RMSE: 7.79
  RF MAE: 5.28
  RF RMSE: 7.97
Total accounted for: 1188 vs. len(X_test): 1188


## Model Comparison & Segment Analysis

**Overall performance (full test set, n=1188):**

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear regression | 5.05 | 8.22 | 0.29 |
| Random forest (tuned, max_depth=8) | 5.08 | 8.19 | 0.29 |

**Segment analysis (absent_rate threshold: 8%):**

| Segment | Model | MAE | RMSE |
|---|---|---|---|
| Low absenteeism (<8%, n=214) | Linear | 4.13 | 9.94 |
| Low absenteeism (<8%, n=214) | RF (tuned) | 4.17 | 9.10 |
| High absenteeism (≥8%, n=974) | Linear | 5.26 | 7.79 |
| High absenteeism (≥8%, n=974) | RF (tuned) | 5.28 | 7.97 |

**Conclusion:**

I compared a linear regression baseline against a tuned random forest (tuned via GridSearchCV, best max_depth: 8) on absent_rate and log_enrolled predicting grad_rate. The two ended up close overall — RF edged out linear on RMSE (8.19 vs. 8.22), tied on R² (0.29), and came in slightly worse on MAE (5.08 vs. 5.05).

I assumed going in that the random forest's flexibility would show up as an advantage in the noisier, high-absenteeism part of the data, but that's not what happened. RF actually did better in the smaller, low-absenteeism segment (n=214), while linear regression held its own — and was slightly ahead — in the much larger, noisier high-absenteeism segment (n=974). Given how small that low-absenteeism segment is, I'm not fully confident that result would hold up on more data. Instead, the high-absenteeism comparison, on a nearly 5x larger sample, feels like the more trustworthy read.

With performance this close and no clear evidence the random forest is actually better suited to the noisy part of the data, I'm going with linear regression. It's simpler, easier to explain, and there's no real upside to the added complexity here.